## Text input

https://platform.openai.com/docs/models

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="gemini-2.5-flash",
    model_provider="google_genai",
)

agent = create_agent(
    model=model,
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
)

In [ ]:
from langchain.messages import HumanMessage

question = HumanMessage(content=[
    {"type": "text", "text": "What is the capital of The Moon?"}
])

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

## Image input

In [5]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [6]:
print(uploader.value)

({'name': 'p02wxbkc.jpg', 'type': 'image/jpeg', 'size': 32328, 'content': <memory at 0x771ff72eb940>, 'last_modified': datetime.datetime(2026, 5, 16, 19, 11, 50, 485000, tzinfo=datetime.timezone.utc)},)


In [7]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [9]:
from langchain.messages import HumanMessage

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this capital"},
    {"type": "image", "base64": img_b64, "mime_type": "image/jpg"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

Welcome to **Lunaris Major**, the gleaming jewel of the Terran-Lunar Alliance, and the undisputed capital of humanity's dominion beyond Earth. Perched strategically on a raised plateau overlooking the vast, ancient Mare Crisium, it stands as a testament to ingenuity, resilience, and the relentless human drive to touch the stars.

At its heart lies the colossal **Aegis Dome**, a magnificent, multi-tiered structure that dominates the lunar horizon. Its curved, iridescent durasteel shell, reinforced with advanced micro-lattice technology, glows with a thousand internal lights, a man-made star against the eternal velvet backdrop of the cosmos. Within this self-contained world, life thrives. The Aegis Dome houses the Alliance Council chambers, the central administrative hub for all lunar and orbital colonies, and the primary diplomatic nexus for interplanetary relations. But it's more than just a government building; it's a vibrant micro-ecosystem, complete with sprawling hydroponic farms t

## Audio input

In [ ]:
import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm

# Recording settings
duration = 5  # seconds
sample_rate = 44100

print("Recording...")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)
# Progress bar for the duration
for _ in tqdm(range(duration * 10)):   # update 10× per second
    time.sleep(0.1)
sd.wait()
print("Done.")

# Write WAV to an in-memory buffer
buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

In [ ]:
agent = create_agent(
    model='gpt-4o-audio-preview',
)

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this audio file"},
    {"type": "audio", "base64": aud_b64, "mime_type": "audio/wav"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)